# 📏 Evaluation Basics

**How to evaluate LLM outputs**

## 📋 Overview

**What you'll learn:**
- Why evaluation matters
- Evaluation metrics
- Ground truth datasets
- Automated vs human eval

**Time estimate:** ⏱️ 50 minutes | **Difficulty:** 🟡 Intermediate

## 🤔 Why Evaluate?

### The Problem:
```
You: "Is my LLM working well?"
❓ How do you know?

- Outputs look good to you
- But are they actually correct?
- Are they better than before?
- How to compare models?
```

### The Solution: Evaluation
```
✅ Measure quality objectively
✅ Compare different approaches
✅ Track improvements over time
✅ Catch regressions
```

## 📊 Evaluation Metrics

### 1. Exact Match
```python
def exact_match(prediction: str, ground_truth: str) -> float:
    """Binary: 1 if exact match, 0 otherwise."""
    return 1.0 if prediction.strip() == ground_truth.strip() else 0.0

# Example
exact_match("Paris", "Paris")  # 1.0
exact_match("Paris", "paris")  # 0.0 (case sensitive)
```

### 2. F1 Score
```python
def f1_score(prediction: str, ground_truth: str) -> float:
    """Token overlap F1 score."""
    pred_tokens = set(prediction.lower().split())
    true_tokens = set(ground_truth.lower().split())
    
    if len(pred_tokens) == 0 or len(true_tokens) == 0:
        return 0.0
    
    common = pred_tokens & true_tokens
    
    precision = len(common) / len(pred_tokens)
    recall = len(common) / len(true_tokens)
    
    if precision + recall == 0:
        return 0.0
    
    return 2 * (precision * recall) / (precision + recall)

# Example
f1_score(
    "The capital of France is Paris",
    "Paris is the capital of France"
)  # ~0.89
```

### 3. BLEU Score (Translation)
```python
from nltk.translate.bleu_score import sentence_bleu

reference = [['the', 'cat', 'sat', 'on', 'the', 'mat']]
candidate = ['the', 'cat', 'is', 'on', 'the', 'mat']

bleu = sentence_bleu(reference, candidate)
print(f"BLEU: {bleu}")  # ~0.75
```

### 4. Semantic Similarity
```python
from openai import OpenAI
import numpy as np

client = OpenAI()

def semantic_similarity(text1: str, text2: str) -> float:
    """Cosine similarity between embeddings."""
    
    # Get embeddings
    emb1 = client.embeddings.create(
        model="text-embedding-3-small",
        input=text1
    ).data[0].embedding
    
    emb2 = client.embeddings.create(
        model="text-embedding-3-small",
        input=text2
    ).data[0].embedding
    
    # Cosine similarity
    return np.dot(emb1, emb2) / (
        np.linalg.norm(emb1) * np.linalg.norm(emb2)
    )

# Example
semantic_similarity(
    "The capital of France is Paris",
    "Paris is the capital of France"
)  # ~0.98 (very similar)
```

## 📝 Creating Evaluation Datasets

```python
# evaluation_set.json
[
    {
        "input": "What is the capital of France?",
        "expected_output": "Paris",
        "category": "geography"
    },
    {
        "input": "Translate to French: Hello",
        "expected_output": "Bonjour",
        "category": "translation"
    },
    {
        "input": "What is 2+2?",
        "expected_output": "4",
        "category": "math"
    }
]
```

### Running Evaluation
```python
import json

def evaluate_model(model_name: str, eval_data: list):
    """Evaluate model on dataset."""
    
    results = []
    
    for item in eval_data:
        # Get model response
        response = client.chat.completions.create(
            model=model_name,
            messages=[{"role": "user", "content": item["input"]}]
        )
        
        prediction = response.choices[0].message.content
        
        # Calculate metrics
        exact = exact_match(prediction, item["expected_output"])
        f1 = f1_score(prediction, item["expected_output"])
        
        results.append({
            "input": item["input"],
            "prediction": prediction,
            "expected": item["expected_output"],
            "exact_match": exact,
            "f1": f1
        })
    
    # Aggregate scores
    avg_exact = np.mean([r["exact_match"] for r in results])
    avg_f1 = np.mean([r["f1"] for r in results])
    
    return {
        "model": model_name,
        "exact_match": avg_exact,
        "f1": avg_f1,
        "results": results
    }
```

## ✅ Summary

**Key takeaways:**

1. **Always evaluate** - Don't trust vibes
2. **Use multiple metrics** - No single metric is perfect
3. **Build eval datasets** - Collect real examples
4. **Track over time** - Monitor regressions

**Evaluation types:**
- **Exact match**: Binary, strict
- **F1 score**: Token overlap
- **Semantic similarity**: Meaning-based
- **Human eval**: Gold standard

**Next steps:**
- Collect diverse test cases
- Run evals on every change
- Compare different prompts
- Track metrics over time

### Next: `12_evaluation/02_model_evaluation.ipynb`